# Disease Embedding Clustering

Extract the learned **word (token) embeddings** from the trained BEHRT model and visualise how diseases cluster in that space.

Each disease code has a 288-dim embedding vector the model learned during pre-training. We reduce to 2-D with UMAP and plot interactively.

In [ ]:
import pickle
import numpy as np
import torch
import os

import umap
import plotly.express as px
import pandas as pd

DATA_DIR   = 'data/behrt'
CKPT_PATH  = os.path.join(DATA_DIR, 'output', 'behrt_best.pt')

## 1. Load vocabulary

In [ ]:
with open(os.path.join(DATA_DIR, 'token2idx.pkl'), 'rb') as f:
    token2idx = pickle.load(f)

idx2token = {v: k for k, v in token2idx.items()}

SPECIAL_TOKENS = {'PAD', 'CLS', 'SEP', 'MASK', 'UNK'}
special_ids    = {token2idx[t] for t in SPECIAL_TOKENS if t in token2idx}

medical_ids   = sorted([idx for idx in idx2token if idx not in special_ids])
medical_names = [idx2token[i] for i in medical_ids]

print(f'Vocabulary size : {len(token2idx)}')
print(f'Special tokens  : {len(special_ids)}')
print(f'Medical codes   : {len(medical_ids)}')
print(f'Sample codes    : {medical_names[:5]}')

## 2. Extract token embeddings from checkpoint

In [ ]:
# ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
# state_dict = ckpt.get('model_state_dict', ckpt)
# 
# Word embedding weight: shape [vocab_size, hidden_size]
# emb_weight   = state_dict['bert.embeddings.word_embeddings.weight'].float().numpy()
# print(f'Full embedding matrix shape: {emb_weight.shape}')
# 
# medical_embs = emb_weight[medical_ids]   # [n_diseases, 288]
# print(f'Medical code embeddings shape: {medical_embs.shape}')

## 3. Reduce to 2-D with UMAP

In [ ]:
# from sklearn.preprocessing import normalize
# from sklearn.decomposition import PCA
# 
# L2 normalize
# medical_embs_normed = normalize(medical_embs)
# 
# PCA: 288 -> 50
# pca = PCA(n_components=50, random_state=42)
# medical_embs_pca = pca.fit_transform(medical_embs_normed)
# print(f'Variance explained by 50 PCs: {pca.explained_variance_ratio_.sum():.1%}')
# 
# UMAP
# reducer = umap.UMAP(
#     n_components=2,
#     n_neighbors=10,
#     min_dist=0.05,
#     metric='euclidean',
#     random_state=42,
#     verbose=True,
# )
# coords_2d = reducer.fit_transform(medical_embs_pca)
# print(f'UMAP output shape: {coords_2d.shape}')

## 4. Interactive scatter plot

In [ ]:
# df = pd.DataFrame({
#     'x':       coords_2d[:, 0],
#     'y':       coords_2d[:, 1],
#     'disease': medical_names,
# })
# 
# fig = px.scatter(
#     df, x='x', y='y',
#     hover_name='disease',
#     title='BEHRT Disease Token Embeddings - UMAP (cosine)',
#     width=1000, height=750,
#     opacity=0.6,
#     template='plotly_white',
# )
# fig.update_traces(marker=dict(size=4))
# fig.show()

## 5. Highlight disease categories of interest

In [ ]:
# Edit keywords to highlight disease groups you care about
# HIGHLIGHT_KEYWORDS = [
#     'hypertension', 'diabetes', 'fracture', 'neoplasm',
#     'cardiac', 'depression', 'hyperlipidemia', 'asthma',
# ]
# 
# def assign_group(name):
#     name_lower = name.lower()
#     for kw in HIGHLIGHT_KEYWORDS:
#         if kw in name_lower:
#             return kw
#     return 'other'
# 
# df['group'] = df['disease'].apply(assign_group)
# 
# color_map = {kw: px.colors.qualitative.Bold[i % 10]
#              for i, kw in enumerate(HIGHLIGHT_KEYWORDS)}
# color_map['other'] = 'lightgrey'
# 
# fig2 = px.scatter(
#     df, x='x', y='y',
#     color='group',
#     hover_name='disease',
#     color_discrete_map=color_map,
#     title='BEHRT Disease Embeddings - highlighted by disease category',
#     width=1000, height=750,
#     opacity=0.7,
#     template='plotly_white',
# )
# fig2.update_traces(marker=dict(size=5))
# fig2.for_each_trace(lambda t: t.update(marker_size=8) if t.name != 'other' else None)
# fig2.show()

## 6. Nearest neighbours for a specific disease

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity
# 
# QUERY = 'Essential hypertension'   # <-- change this
# TOP_K = 20
# 
# if QUERY in token2idx:
#     q_idx = medical_ids.index(token2idx[QUERY])
#     q_vec = medical_embs[q_idx].reshape(1, -1)
#     sims  = cosine_similarity(q_vec, medical_embs)[0]
#     top_k = np.argsort(sims)[::-1][1:TOP_K + 1]
# 
#     print(f"Top {TOP_K} diseases most similar to '{QUERY}':")
#     for rank, i in enumerate(top_k, 1):
#         print(f'  {rank:>2}. {medical_names[i]:<60s}  sim={sims[i]:.4f}')
# else:
#     print(f"'{QUERY}' not found. Sample names: {medical_names[:10]}")

---
## Option 2: Contextualized disease embeddings (patient centroid averaging)

For each disease, find all patients who have it in their code sequence and average their 288-dim patient embeddings.
These are the pooled CLS embeddings from `patient_embeddings.npy` — fully contextualized after all 7 transformer layers.

We memory-map the 13 GB file so it is never fully loaded into RAM.

In [ ]:
import numpy as np
import pickle
import os
from collections import defaultdict
from tqdm.auto import tqdm

EMB_PATH          = os.path.join(DATA_DIR, 'output', 'patient_embeddings.npy')
SEQ_PATH          = os.path.join(DATA_DIR, 'patient_sequences.pkl')
D2R_CACHE_PATH    = os.path.join(DATA_DIR, 'output', 'disease_to_rows.pkl')

# Memory-map embeddings -- instant, no data read yet
emb_mmap = np.load(EMB_PATH, mmap_mode='r')
print(f'Embeddings shape: {emb_mmap.shape}')

# Load disease->row index mapping from cache if available
# (first run builds it from sequences and saves it; subsequent runs skip the 21 GB load)
if os.path.exists(D2R_CACHE_PATH):
    print('Loading cached disease_to_rows...')
    with open(D2R_CACHE_PATH, 'rb') as f:
        disease_to_rows = pickle.load(f)
    print(f'Loaded {len(disease_to_rows):,} diseases from cache')
else:
    import pandas as pd
    print('Cache not found. Loading sequences (slow, one-time only)...')
    sequences = pd.read_pickle(SEQ_PATH)
    print(f'Sequences loaded: {sequences.shape}')

    disease_to_rows = defaultdict(list)
    for row_idx, code_list in enumerate(tqdm(sequences['code'], desc='Indexing')):
        seen = set(code_list) - {'SEP'}
        for disease in seen:
            if disease in token2idx and token2idx[disease] not in special_ids:
                disease_to_rows[disease].append(row_idx)

    # Sort indices for sequential mmap access later
    for d in disease_to_rows:
        disease_to_rows[d].sort()

    with open(D2R_CACHE_PATH, 'wb') as f:
        pickle.dump(dict(disease_to_rows), f)
    print(f'Cached {len(disease_to_rows):,} diseases to {D2R_CACHE_PATH}')

    del sequences

In [ ]:
# Average patient embeddings per disease using a single sequential pass
# (reads emb_mmap row-by-row in order, far faster than random index access)
MIN_PATIENTS = 10

disease_list = [d for d, rows in disease_to_rows.items() if len(rows) >= MIN_PATIENTS]
d_to_i       = {d: i for i, d in enumerate(disease_list)}
hidden       = emb_mmap.shape[1]

sums   = np.zeros((len(disease_list), hidden), dtype=np.float64)
counts = np.zeros(len(disease_list), dtype=np.int64)

# One sequential pass over rows
CHUNK = 100_000
n_rows = emb_mmap.shape[0]

# Build reverse map: row_idx -> list of disease indices
print('Building row->disease index...')
row_to_diseases = defaultdict(list)
for disease, rows in disease_to_rows.items():
    if disease in d_to_i:
        for r in rows:
            row_to_diseases[r].append(d_to_i[disease])

print('Accumulating embeddings...')
for start in tqdm(range(0, n_rows, CHUNK), desc='Chunks'):
    end   = min(start + CHUNK, n_rows)
    chunk = emb_mmap[start:end].astype(np.float64)  # sequential read
    for local_i, row_idx in enumerate(range(start, end)):
        for d_idx in row_to_diseases.get(row_idx, []):
            sums[d_idx]   += chunk[local_i]
            counts[d_idx] += 1

disease_names_ctx  = disease_list
disease_embs_ctx   = (sums / np.maximum(counts[:, None], 1)).astype(np.float32)
disease_counts_ctx = counts.tolist()

print(f'Diseases retained: {len(disease_names_ctx)}')
print(f'Centroid matrix shape: {disease_embs_ctx.shape}')


In [ ]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

# L2 normalize
disease_embs_normed = normalize(disease_embs_ctx)

# PCA: 288 -> 50
pca_ctx = PCA(n_components=50, random_state=42)
disease_embs_pca = pca_ctx.fit_transform(disease_embs_normed)
print(f'Variance explained by 50 PCs: {pca_ctx.explained_variance_ratio_.sum():.1%}')

# UMAP
import umap
reducer_ctx = umap.UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.05,
    metric='euclidean',
    random_state=42,
    verbose=True,
)
coords_ctx = reducer_ctx.fit_transform(disease_embs_pca)
print(f'UMAP output shape: {coords_ctx.shape}')


In [ ]:
from sklearn.cluster import KMeans

N_CLUSTERS = 15  # adjust as needed

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(coords_ctx)

df_ctx = pd.DataFrame({
    'x':          coords_ctx[:, 0],
    'y':          coords_ctx[:, 1],
    'disease':    disease_names_ctx,
    'n_patients': disease_counts_ctx,
    'cluster':    [str(c) for c in cluster_labels],
})

fig3 = px.scatter(
    df_ctx, x='x', y='y',
    color='cluster',
    hover_name='disease',
    hover_data={'n_patients': True, 'x': False, 'y': False, 'cluster': False},
    size='n_patients',
    size_max=20,
    title='BEHRT Contextualized Disease Embeddings -- patient centroid UMAP (cosine)',
    width=1100, height=800,
    opacity=0.7,
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig3.show()
fig3.write_html("disease_umap.html") 